In [1]:
import os
from pyspark.sql import SparkSession

# Define the required packages for Iceberg and S3 (MinIO) connectivity
packages = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4"
]

spark = SparkSession.builder \
    .appName("UnifiedBharat_Pipeline") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://unified-bharat/warehouse") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "supersecretpassword") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Running. Version:", spark.version)

Spark Running. Version: 3.5.0


In [17]:
lgd_df = spark.read.csv("../data/raw/LGD.csv", header=True, inferSchema=True)
csr_df = spark.read.csv("../data/raw/csr_district.csv", header=True, inferSchema=True)
water_df = spark.read.csv("../data/raw/ground_water.csv", header=True, inferSchema=True)

In [18]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

csr_df.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("local.bronze.csr_spending")

water_df.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("local.bronze.water_df")

lgd_df.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable("local.bronze.lgd_df")

print("Table successfully registered in Iceberg!")

Table successfully registered in Iceberg!


In [19]:
csr_df.write.format("iceberg").mode("overwrite").save("local.bronze.csr_spending")
water_df.write.format("iceberg").mode("overwrite").save("local.bronze.water_df")
lgd_df.write.format("iceberg").mode("overwrite").save("local.bronze.lgd_df")

In [20]:
spark.read.format("iceberg").load("local.bronze.csr_spending").show(5)

+-------+-----------+---------+--------------------+------------+--------------------+--------------------+--------------------+-----------+-----------+-----------+-----------+-------------+--------------+----------------------+-----------------------------+--------------------------------+---------------------------------+-----------------------------------+
|Country|  StateName|StateCode|        DistrictName|DistrictCode|                Year|             D6008_4|             D6008_5|I6008_6.avg|I6008_6.max|I6008_6.min|I6008_6.sum|I6008_6.count|I6008_6.stddev|I6008_6.LandAreaWeight|I6008_6.TotalPopulationWeight|I6008_6.NumberOfHouseholdsWeight|I6008_6.TotalPopulationMaleWeight|I6008_6.TotalPopulationFemaleWeight|
+-------+-----------+---------+--------------------+------------+--------------------+--------------------+--------------------+-----------+-----------+-----------+-----------+-------------+--------------+----------------------+-----------------------------+------------------